# Setup — Primera exploración

Carga de los 9 CSV's de Olist a la capa 'bronze' de DuckDB, y exploración del dataset

In [1]:
import duckdb
from pathlib import Path

DATA_DIR = Path("../../Data") # carpeta con mis CSVs
DB_PATH = Path("../olist.duckdb") 

tablas = {
    "orders": "olist_orders_dataset",
    "order_items": "olist_order_items_dataset",
    "customers": "olist_customers_dataset",
    "products": "olist_products_dataset",
    "sellers": "olist_sellers_dataset",
    "payments": "olist_order_payments_dataset",
    "reviews": "olist_order_reviews_dataset",
    "geolocation": "olist_geolocation_dataset",
    "category_translation": "product_category_name_translation",
}

con = duckdb.connect(str(DB_PATH))
con.execute("CREATE SCHEMA IF NOT EXISTS bronze")

for tabla, archivo in tablas.items():
    csv = DATA_DIR / f"{archivo}.csv"
    assert csv.exists(), f"No encuentro el CSV: {csv.resolve()}"
    con.execute(f"""
        CREATE OR REPLACE TABLE bronze.{tabla} AS
        SELECT * FROM read_csv_auto('{csv.as_posix()}')
    """)

    n = con.execute(f"SELECT count(*) FROM bronze.{tabla}").fetchone()[0]
    print(f" bronze.{tabla:22s} <- {archivo}.csv({n:,} filas)")

con.close()
print("Carga completa.")

 bronze.orders                 <- olist_orders_dataset.csv(99,441 filas)
 bronze.order_items            <- olist_order_items_dataset.csv(112,650 filas)
 bronze.customers              <- olist_customers_dataset.csv(99,441 filas)
 bronze.products               <- olist_products_dataset.csv(32,951 filas)
 bronze.sellers                <- olist_sellers_dataset.csv(3,095 filas)
 bronze.payments               <- olist_order_payments_dataset.csv(103,886 filas)
 bronze.reviews                <- olist_order_reviews_dataset.csv(99,224 filas)
 bronze.geolocation            <- olist_geolocation_dataset.csv(1,000,163 filas)
 bronze.category_translation   <- product_category_name_translation.csv(71 filas)
Carga completa.


In [2]:
con = duckdb.connect(str(DB_PATH), read_only=True)
print(con.execute("SELECT COUNT(*) FROM bronze.orders").fetchone()[0])
con.execute("SHOW TABLES").df()     # lista las tablas del bronze

99441


,name


## Conociendo la data

### Esquema y tipos

Qué columnas tiene cada tabla y qué tipo les infirió `read_csv_auto`

In [3]:
con.execute("DESCRIBE bronze.orders").df()

,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,customer_id,VARCHAR,YES,None,None,None
2,order_status,VARCHAR,YES,None,None,None
3,order_purchase_timestamp,TIMESTAMP,YES,None,None,None
4,order_approved_at,TIMESTAMP,YES,None,None,None
5,order_delivered_carrier_date,TIMESTAMP,YES,None,None,None
6,order_delivered_customer_date,TIMESTAMP,YES,None,None,None
7,order_estimated_delivery_date,TIMESTAMP,YES,None,None,None


### Rango de fechas

Ver de dónde a dónde va el dataset → definir el eje de las cohortes

In [4]:
con.execute("""
    SELECT min(order_purchase_timestamp), max(order_purchase_timestamp)
    FROM bronze.orders
""").df()

,min(order_purchase_timestamp),max(order_purchase_timestamp)
0,2016-09-04 21:15:19,2018-10-17 17:30:18


### Nulls

Ver dónde faltan datos. Ej. en `orders` las fechas de entrega tienen nulls

In [5]:
con.execute("""
    SELECT
        count(*) AS total,
        count(order_delivered_customer_date) AS con_entrega,
        count(*) - count(order_delivered_customer_date) AS sin_entrega
    FROM bronze.orders
""").df()

,total,con_entrega,sin_entrega
0,99441,96476,2965


### Identidad: customer_unique_id

Observamos que hay dos diferentes: `customer_id` y `customer_unique_id`

In [6]:
con.execute("""
    SELECT
        count(*) AS filas,
        count(DISTINCT customer_id) AS ids_por_orden,
        count(DISTINCT customer_unique_id) AS personas_reales
    FROM bronze.customers
""").df()

,filas,ids_por_orden,personas_reales
0,99441,99441,96096


### Duplicados

Analizar las PKs. e.g. que `order_items` es único por (`order_id`, `order_item_id`) y que `orders.order_id` no tiene duplicados.

In [7]:
con.execute("""
    SELECT
        count(*) AS filas,
        count(DISTINCT order_id) AS ordenes_unicas
    FROM bronze.orders
""").df()

,filas,ordenes_unicas
0,99441,99441


**Tarea 3 del plan — 16 queries de negocio sobre `marts`, agrupadas en 6 bloques.**

## Bloque 1 — Volumen y tendencia

### 1.1 Órdenes y revenue por mes

In [8]:
con.execute("""
    select
        strftime(fecha_compra, '%Y-%m') as anio_mes,
        count(*)                        as cantidad_ordenes,
        round(sum(valor_total), 2)      as revenue_total
    from marts.fct_ordenes
    group by anio_mes
    order by anio_mes
""").df()

,anio_mes,cantidad_ordenes,revenue_total
0,2016-09,4,354.75
1,2016-10,324,56808.84
2,2016-12,1,19.62
3,2017-01,800,137188.49
4,2017-02,1780,286280.62
5,2017-03,2682,432048.59
6,2017-04,2404,412422.24
7,2017-05,3700,586190.95
8,2017-06,3245,502963.04
9,2017-07,4026,584971.62


### 1.2 Distribución de `order_status`

In [9]:
con.execute("""
    select
        order_status,
        count(*) as cantidad_ordenes,
        round(100*count(*) / sum(count(*)) over (), 2) as porcentaje
    from marts.fct_ordenes
    group by order_status
    order by cantidad_ordenes desc
""").df()

,order_status,cantidad_ordenes,porcentaje
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


### 1.3 Ticket promedio por mes

In [10]:
con.execute("""
    select
        strftime(fecha_compra, '%Y-%m') as anio_mes,
        round(avg(valor_total), 2) as ticket_promedio
    from marts.fct_ordenes
    where valor_total > 0
    group by anio_mes
    order by anio_mes
""").df()

,anio_mes,ticket_promedio
0,2016-09,118.25
1,2016-10,184.44
2,2016-12,19.62
3,2017-01,173.88
4,2017-02,165.19
5,2017-03,163.59
6,2017-04,172.49
7,2017-05,160.16
8,2017-06,156.35
9,2017-07,147.39


obs $\longrightarrow$ Vemos consistencia en los datos. Notamos que `2018-10` no aparece en la tabla: es consecuencia de que en ese mes se tiene `valor_total = 0`, así que nuestro filtro `where valor_total>0` las descarta y no queda filas para promediar.

## Bloque 2 — Producto y categoría

### 2.1 Top 10 categorías por revenue

In [11]:
con.execute("""
    select
        p.product_category_name_english as categoria,
        round(sum(i.valor_total), 2) as revenue_total
    from marts.fct_items i
    left join marts.dim_producto p on i.product_id = p.product_id
    group by categoria
    order by revenue_total desc
    limit 10
""").df()

,categoria,revenue_total
0,health_beauty,1441248.07
1,watches_gifts,1305541.61
2,bed_bath_table,1241681.72
3,sports_leisure,1156656.48
4,computers_accessories,1059272.40
5,furniture_decor,902511.79
6,housewares,778397.77
7,cool_stuff,719329.95
8,auto,685384.32
9,garden_tools,584219.21


### 2.2 Top 10 categorías por items vendidos

Ahora veremos no cuánto dinero, sino cuántas unidades. Observaciones: Puede o no coincidir con 2.1 - Una categoría puede vender mucho en volumen pero ítems baratos.

In [12]:
con.execute("""
    select
        p.product_category_name_english as categoria,
        count(*) as cantidad_items
    from marts.fct_items i
    left join marts.dim_producto p on i.product_id = p.product_id
    group by categoria
    order by cantidad_items desc
    limit 10
""").df()

,categoria,cantidad_items
0,bed_bath_table,11115
1,health_beauty,9670
2,sports_leisure,8641
3,furniture_decor,8334
4,computers_accessories,7827
5,housewares,6964
6,watches_gifts,5991
7,telephony,4545
8,garden_tools,4347
9,auto,4235


### 2.3 Valor promedio por ítem, por categoría (Top 10 más caras)

Si una categoría solo tiene 3 items vendidos en total, su promedio podría ser engañoso (Un solo ítem caro lo dispara). Entonces vamos a pedir `having count(*) >= 30` - Exigimos un mínimo de muestras para ver los promedios

In [13]:
con.execute("""
    select
        p.product_category_name_english as categoria,
        round(avg(i.valor_total), 2) as valor_promedio_item,
        count(*) as cantidad_items
    from marts.fct_items i
    left join marts.dim_producto p on i.product_id = p.product_id
    group by categoria
    having count(*) >= 30
    order by valor_promedio_item desc
    limit 10
""").df()

,categoria,valor_promedio_item,cantidad_items
0,computers,1146.80,203
1,small_appliances_home_oven_and_coffee,660.44,76
2,home_appliances_2,520.66,238
3,agro_industry_and_commerce,369.69,212
4,musical_instruments,309.03,680
5,small_appliances,304.37,679
6,fixed_telephony,243.26,264
7,construction_tools_safety,229.19,194
8,furniture_bedroom,226.25,109
9,watches_gifts,217.92,5991


### 2.4 Porcentaje de ítems vendidos en sin_categoria. 

En `dim_producto` etiquetamos como `sin_categoria` los productos sin traducción, en lugar de dejarlos como `null`. Acá mediremos cuánto pesa eso a nivel de ítems vendidos

In [14]:
con.execute("""
    select
        count(*) as total_items,
        sum(case when p.product_category_name_english = 'sin_categoria' then 1 else 0 end) as items_sin_categoria,
        round(100.0 * sum(case when p.product_category_name_english = 'sin_categoria' then 1 else 0 end) / count(*) , 2) as porcentaje
    from marts.fct_items i
    left join marts.dim_producto p on i.product_id = p.product_id
""").df()

,total_items,items_sin_categoria,porcentaje
0,112650,1627.0,1.44


## Bloque 3 — Cliente

### 3.1 Porcentaje de clientes con más de una orden. 

Agrupar `fct_ordenes` por `customer_unique_id` para contar órdenes por persona, y sobre eso medir el porcentaje que tiene una `cantidad_ordenes > 1`

In [15]:
con.execute("""
    with ordenes_por_cliente as (
        select
            customer_unique_id, 
            count(*) as cantidad_ordenes
        from marts.fct_ordenes
        group by customer_unique_id
    )
    
    select
        count(*)    as total_clientes,
        sum(case when cantidad_ordenes > 1 then 1 else 0 end)   as clientes_recompra,
        round(100.0 * sum(case when cantidad_ordenes > 1 then 1 else 0 end) / count(*), 2)  as porcentaje_recompra
    from ordenes_por_cliente
""").df()

,total_clientes,clientes_recompra,porcentaje_recompra
0,96096,2997.0,3.12


### 3.2 Tiempo promedio entre 1ª y 2ª compra

A diferencia del `over()` vacío del bloque 1 (una sola ventana todo el resultado), acá `partition by` divide la ventana por cliente -- dentro de cada cliente numera sus órdenes en orden cronológico (1ª, 2ª, 3ª...). Con esto se sepera la primera orden de la segunda, y las unimos para calcular la diferencia de días.

Usamos `inner join`, ya que queremos quedarnos solo con los clientes que sí tienen una 2ª orden

In [16]:
con.execute("""
    with ordenes_numeradas as (
        select
            customer_unique_id,
            fecha_compra,
            row_number() over (partition by customer_unique_id order by fecha_compra) as rn
        from marts.fct_ordenes
    ),
    primera as (
        select
            customer_unique_id,
            fecha_compra as fecha_1ra_compra
        from ordenes_numeradas where rn = 1
    ),
    segunda as (
        select
            customer_unique_id,
            fecha_compra as fecha_2da_compra
        from ordenes_numeradas where rn = 2
    )
    
    select
        count(*)    as clientes_con_recompra,
        round(avg(date_diff('day', p.fecha_1ra_compra, s.fecha_2da_compra)),1) as dias_promedio_1ra_2da
    from primera p
    inner join segunda s on p.customer_unique_id = s.customer_unique_id
""").df()

,clientes_con_recompra,dias_promedio_1ra_2da
0,2997,80.3


### 3.3 Distribución de cantidad de órdenes por cliente.

Para ver la forma completa, no solo el promedio: ¿la mayoría tiene 1 sola orden y hay una cola larga de gente con 2, 3, 10+?

In [17]:
con.execute("""
    with ordenes_por_cliente as (
        select
            customer_unique_id,
            count(*) as cantidad_ordenes
        from marts.fct_ordenes
        group by customer_unique_id
    )
            
    select
        cantidad_ordenes,
        count(*) as cantidad_clientes
    from ordenes_por_cliente
    group by cantidad_ordenes
    order by cantidad_ordenes
""").df()

,cantidad_ordenes,cantidad_clientes
0,1,93099
1,2,2745
2,3,203
3,4,30
4,5,8
5,6,6
6,7,3
7,9,1
8,17,1


## Bloque 4 — Geografía

### 4.1 Top 10 estados por cantidad de clientes

`dim_cliente` tiene `customer_state` como atributo propio de cada persona

In [18]:
con.execute("""
    select
        customer_state,
        count(*) as cantidad_clientes
    from marts.dim_cliente
    group by customer_state
    order by cantidad_clientes desc
    limit 10
""").df()

,customer_state,cantidad_clientes
0,SP,40291
1,RJ,12379
2,MG,11255
3,RS,5277
4,PR,4881
5,SC,3531
6,BA,3276
7,DF,2074
8,ES,1964
9,GO,1950


### 4.2 Top 10 estados por revenue

Acá haremos un join: el revenue (`valor_total`) vive en `fct_ordenes`, pero el estado (el state) vive en `dim_cliente`

In [19]:
con.execute("""
    select
        c.customer_state,
        round(sum(o.valor_total), 2) as revenue_total
    from marts.fct_ordenes o
    left join marts.dim_cliente c on o.customer_unique_id = c.customer_unique_id
    group by c.customer_state
    order by revenue_total desc
    limit 10
""").df()

,customer_state,revenue_total
0,SP,5921600.60
1,RJ,2129102.67
2,MG,1856503.69
3,RS,886076.83
4,PR,801323.68
5,BA,611457.60
6,SC,610092.83
7,DF,353329.74
8,GO,347372.55
9,ES,324946.11


## Bloque 5 — Vendedores

### 5.1 Top 10 sellers por revenue

In [20]:
con.execute("""
    select
        seller_id,
        round(sum(valor_total), 2) as revenue_total,
        count(*)        as cantidad_items
    from marts.fct_items
    group by seller_id
    order by revenue_total desc
    limit 10
""").df()

,seller_id,revenue_total,cantidad_items
0,4869f7a5dfa277a7dca6462dcf3b52b2,249640.70,1156
1,7c67e1448b00f6e969d365cea6b010ab,239536.44,1364
2,53243585a1d6dc2643021fd1853d8905,235856.68,410
3,4a3ca9315b744ce9f8e9374361493884,235539.96,1987
4,fa1c13f2614d7b5c4749cbc52fecda94,204084.73,586
5,da8622b14eb17ae2831f4ac5b9dab84a,185192.32,1551
6,7e93a43ef30c4f03f38b393420bc753a,182754.05,340
7,1025f0e2d44d7041d6cf58b6550e0bfa,172860.69,1428
8,7a67c85e85bb2ce8582c35f2203ad736,162648.38,1171
9,955fee9216a65b617aa5c0531780ce60,160602.68,1499


### 5.2 Cantidad de sellers activos

¿Cuántos vendedores distintos hay realmente vendiendo? (i.e. no todo el catálogo de sellers, sino los que aparecen en al menos una venta)

obs: `fct_items` no es un catálogo, es un registro de ventas.

Cada fila de esa tabla ya significa "esto se vendió, en tal orden, a tal cliente". No es una lista de "todos los vendedores que existen" (eso sería `bronze.sellers`)

Es decir, si un vendedor nunca vendió nada, no tendrían ninguna fila en `fct_items`

In [21]:
con.execute("""
    select count(distinct seller_id) as sellers_activos
    from marts.fct_items
""").df()

,sellers_activos
0,3095


## Bloque 6 — Pagos y satisfacción

no van contra `marts`, sino contra `staging` (`stg_olist__payments`, `stg_olist__reviews`) - no hemos construido un mart para pagos/reseñas (no estaba en el alcance de la tarea), así que para explorarlas por ahora hoy 5/07/2026 alcanza con la capa staging.

En caso se vuelva un análisis recurrente (no solo exploración de hoy), ahí valdría hacer un `fct_pagos`

### 6.1 Distribución de `payment_type`

In [22]:
con.execute("""
    select
        payment_type,
        count(*)    as cantidad_pagos,
        round(100.0 * count(*) / sum(count(*)) over (), 2) as porcentaje
    from staging.stg_olist__payments
    group by payment_type
    order by cantidad_pagos desc
""").df()

,payment_type,cantidad_pagos,porcentaje
0,credit_card,76795,73.92
1,boleto,19784,19.04
2,voucher,5775,5.56
3,debit_card,1529,1.47
4,not_defined,3,0.00


### 6.2 Cuotas promedio, por tipo de pago.

En vez de solo un promedio global, lo partimos por `payment_type` -- más informativo

In [23]:
con.execute("""
    select
        payment_type,
        round(avg(payment_installments), 2) as cuotas_promedio,
        max(payment_installments) as cuotas_max
    from staging.stg_olist__payments
    group by payment_type
    order by cuotas_promedio desc
""").df()

,payment_type,cuotas_promedio,cuotas_max
0,credit_card,3.51,24
1,boleto,1.00,1
2,voucher,1.00,1
3,debit_card,1.00,1
4,not_defined,1.00,1


### 6.3 Distribución de `review_score`: Satisfacción del cliente

1 a 5 estrellas

In [24]:
con.execute("""
    select
        review_score,
        count(*)        as cantidad_reviews,
        round(100.0 * count(*) / sum(count(*)) over (), 2) as porcentaje
    from staging.stg_olist__reviews
    group by review_score
    order by review_score
""").df()

,review_score,cantidad_reviews,porcentaje
0,1,11424,11.51
1,2,3151,3.18
2,3,8179,8.24
3,4,19142,19.29
4,5,57328,57.78


---

## Quality Checks — base antes de analizar

Paso entre la Sem1 y Sem2, tenemos que asegurar una base limpia, con la identidad del cliente resuelta

**Tarea 1 -- Resolver la identidad del cliente**

Esto ya lo hicimos, todo el modelo dimensional se contruyó sobre `customer_unique_id`

Pero vamos a dejarlo explícito con resultados cuantitativos.

### QC 1 - Identidad del cliente: `customer_unique_id`, no `customer_id`

`customer_id` es casi 1:1 con las órdenes ($99.441$ valores distintos) - no identifica a la persona real. Todo el análisis del cliente (recompra, cohortes, RFM) va sobre `customer_unique_id`.

In [25]:
con.execute("""
    select
        count(distinct customer_unique_id) as personas_reales,
        (select count(*) from marts.fct_ordenes) as ordenes_totales
    from marts.dim_cliente
""").df()

,personas_reales,ordenes_totales
0,96096,99441


observamos muchas más órdenes que personas reales

**Tarea 2 - Definir la base analítica de forma explícita**

No todas las órdenes sirven para medir el negocio. Nos pidieron filtrar por **canceled/unavailable** y dejarlo documentado.

Ahora, vamos a filtrar por status, no por si tuvo items

### QC 2 - Base analítica: se excluyen 'canceled' y 'unavailable'

Se excluyen las órdenes con `orden_status` en (`canceled`, `unavailable`): no representan una transacción real completa y mezclarlas distorsiona cualquier métrica de negocio (revenue, recompra, tiempo entre compras).

De acá en adelante, todo análisis de negocio parte de `base_analitica`, no de `fct_ordenes` completa.

In [26]:
base_analitica = con.execute("""
    select *
    from marts.fct_ordenes
    where order_status not in ('canceled', 'unavailable')
""").df()

In [27]:
print(f"Órdenes en base_analítica: {len(base_analitica):,}")

Órdenes en base_analítica: 98,207


**Tarea 3 — Reconciliación (funnel de filas)**

Por qué: paso nuevo que no estaba en el plan original - una celda que muestre, en números, cómo baja el conteo en cada filtro. Si alguien pregunta ¿de dónde sale este total?, lo tenemos a mano en una sola query

### QC 3 - Funnel de reconciliación: N órdenes $\rightarrow$ M órdenes $\rightarrow$ K clientes

Con eso tenemos N (todas las órdenes), M (tras el filtro de status) y K (personas únicas detrás de esas M órdenes)

In [28]:
funnel = con.execute("""
    select
        (select count(*) from marts.fct_ordenes)    as n_ordenes_totales,
        count(*)    as m_ordenes_base_analitica,
        count(distinct customer_unique_id)  as k_clientes_unicos
    from marts.fct_ordenes
    where order_status not in ('canceled', 'unavailable')
""").df()
funnel

,n_ordenes_totales,m_ordenes_base_analitica,k_clientes_unicos
0,99441,98207,94990


**Tarea 4 -- Una sola FECHA_CORTE**

Por qué: Se define de una vez, como el máximo de fecha dentro de la base analítica (no `today()` - no tiene sentido medir recency contra la fecha de hoy en 2026 sobre un dataset que termina en 2018), y esa misma variable se va a reutilizar en Sem. 2 y 3

### QC 4 - `FECHA_CORTE`: fecha de corte única para Recency, cohortes y LTV

se define como el máximo de `fecha_compra` dentro de la base analítica

In [29]:
FECHA_CORTE = con.execute("""
    select
        max(fecha_compra)
    from marts.fct_ordenes
    where order_status not in ('canceled', 'unavailable')
""").fetchone()[0]

In [30]:
print(f"FECHA_CORTE = {FECHA_CORTE}")

FECHA_CORTE = 2018-09-03


## Cierre — liberar la conexión

Cerramos `con` para que el `01` no deje `olist.duckdb` tomada: así el `02` y el `03` (y dbt, si se reconstruye) pueden abrirla sin lock.

In [31]:
con.close()
print("Conexión cerrada — olist.duckdb liberada.")

Conexión cerrada — olist.duckdb liberada.
